# 02 — Generate Customer CDC Events

## Purpose

This notebook generates a deterministic batch of customer Change Data Capture events representing changes received from the CRM after the initial customer snapshot.

The generated batch contains INSERT, UPDATE, and DELETE operations and will later be processed incrementally through the Bronze and Silver layers.

## CDC Event Distribution

- 250 UPDATE events
- 200 INSERT events
- 50 DELETE events
- 500 total events

## Data Quality Controls

- Explicit source schema
- Mandatory date and timestamp validation
- Expected event-count validation
- Operation-distribution validation
- Duplicate-event detection
- Business-key validation
- Referential-integrity validation
- Event-sequence validation

## Input

`/Volumes/workspace/revenue_leakage_bronze/landing/crm/customers/initial_load`

## Target

`/Volumes/workspace/revenue_leakage_bronze/landing/crm/customers/change_batch_001`

## 1. Configuration and Source Schema

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    TimestampType,
)


INITIAL_CUSTOMER_COUNT = 5000

UPDATE_COUNT = 250
INSERT_COUNT = 200
DELETE_COUNT = 50

EXPECTED_EVENT_COUNT = (
    UPDATE_COUNT
    + INSERT_COUNT
    + DELETE_COUNT
)

LANDING_PATH = (
    "/Volumes/workspace/"
    "revenue_leakage_bronze/"
    "landing"
)

CUSTOMERS_PATH = f"{LANDING_PATH}/crm/customers"

CUSTOMERS_INITIAL_PATH = (
    f"{CUSTOMERS_PATH}/initial_load"
)

CUSTOMERS_CHANGE_BATCH_PATH = (
    f"{CUSTOMERS_PATH}/change_batch_001"
)

CUSTOMER_SCHEMA = StructType([
    StructField("customer_id", StringType(), False),
    StructField("first_name", StringType(), False),
    StructField("last_name", StringType(), False),
    StructField("email", StringType(), True),
    StructField("country", StringType(), False),
    StructField("region", StringType(), False),
    StructField("customer_segment", StringType(), False),
    StructField("signup_date", DateType(), True),
    StructField("customer_status", StringType(), False),
    StructField("operation", StringType(), False),
    StructField("event_timestamp", TimestampType(), True),
])

## 2. Load and Validate the Initial Snapshot

Load the initial customer snapshot using an explicit schema. Validate the expected row count and confirm that source dates and timestamps were parsed successfully.

In [0]:
customers_initial_df = (
    spark.read
    .schema(CUSTOMER_SCHEMA)
    .json(CUSTOMERS_INITIAL_PATH)
)

initial_count = customers_initial_df.count()

invalid_initial_timestamp_count = (
    customers_initial_df
    .filter(
        F.col("signup_date").isNull()
        | F.col("event_timestamp").isNull()
    )
    .count()
)

assert initial_count == INITIAL_CUSTOMER_COUNT
assert invalid_initial_timestamp_count == 0

assert isinstance(
    customers_initial_df.schema["signup_date"].dataType,
    DateType
)

assert isinstance(
    customers_initial_df.schema["event_timestamp"].dataType,
    TimestampType
)

print(f"Initial customers loaded: {initial_count:,}")

print(
    "Invalid initial dates or timestamps: "
    f"{invalid_initial_timestamp_count:,}"
)

display(
    customers_initial_df
    .orderBy("customer_id")
    .limit(10)
)

customers_initial_df.printSchema()

## 3. Generate UPDATE Events

Generate changes for 250 existing customers. Customer segments are changed deterministically, and selected customer statuses are updated to Inactive.

In [0]:
customer_updates_df = (
    customers_initial_df
    .orderBy("customer_id")
    .limit(UPDATE_COUNT)

    .withColumn(
        "customer_segment",
        F.when(
            F.col("customer_segment") == "Standard",
            "Premium"
        )
        .when(
            F.col("customer_segment") == "Premium",
            "VIP"
        )
        .otherwise("Standard")
    )

    .withColumn(
        "customer_status",
        F.when(
            F.regexp_extract(
                "customer_id",
                r"(\d+)$",
                1
            ).cast("int") % 10 == 0,
            "Inactive"
        ).otherwise(F.col("customer_status"))
    )

    .withColumn(
        "operation",
        F.lit("UPDATE")
    )

    .withColumn(
        "event_timestamp",
        F.to_timestamp(
            F.lit("2026-08-16 10:00:00")
        )
    )
)

## 4. Generate INSERT Events

Generate 200 new customer records whose business keys do not exist in the initial snapshot.

In [0]:
customer_inserts_df = (
    customers_initial_df
    .orderBy("customer_id")
    .limit(INSERT_COUNT)

    .withColumn(
        "new_customer_number",
        F.regexp_extract(
            "customer_id",
            r"(\d+)$",
            1
        ).cast("int") + INITIAL_CUSTOMER_COUNT
    )

    .withColumn(
        "customer_id",
        F.format_string(
            "C%06d",
            F.col("new_customer_number")
        )
    )

    .withColumn(
        "email",
        F.concat(
            F.lower("first_name"),
            F.lit("."),
            F.lower("last_name"),
            F.col("new_customer_number").cast("string"),
            F.lit("@example.com")
        )
    )

    .withColumn(
        "signup_date",
        F.to_date(F.lit("2026-08-16"))
    )

    .withColumn(
        "customer_status",
        F.lit("Active")
    )

    .withColumn(
        "operation",
        F.lit("INSERT")
    )

    .withColumn(
        "event_timestamp",
        F.to_timestamp(
            F.lit("2026-08-16 10:05:00")
        )
    )

    .drop("new_customer_number")
)

## 5. Generate DELETE Events

Generate DELETE events for 50 customers that currently exist in the initial snapshot.

In [0]:
customer_deletes_df = (
    customers_initial_df
    .orderBy(F.desc("customer_id"))
    .limit(DELETE_COUNT)

    .withColumn(
        "operation",
        F.lit("DELETE")
    )

    .withColumn(
        "event_timestamp",
        F.to_timestamp(
            F.lit("2026-08-16 10:10:00")
        )
    )
)

## 6. Combine and Validate the CDC Batch

Combine all CDC operations and validate event counts, key uniqueness, referential integrity, duplicate events, mandatory fields, and chronological sequencing.

In [0]:
customer_changes_df = (
    customer_updates_df
    .unionByName(customer_inserts_df)
    .unionByName(customer_deletes_df)
)

actual_event_count = customer_changes_df.count()

distinct_event_customer_count = (
    customer_changes_df
    .select("customer_id")
    .distinct()
    .count()
)

null_event_key_count = (
    customer_changes_df
    .filter(
        F.col("customer_id").isNull()
        | F.col("operation").isNull()
        | F.col("event_timestamp").isNull()
    )
    .count()
)

duplicate_event_count = (
    customer_changes_df
    .groupBy(
        "customer_id",
        "operation",
        "event_timestamp"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

existing_customer_keys_df = (
    customers_initial_df
    .select("customer_id")
)

missing_update_key_count = (
    customer_updates_df
    .select("customer_id")
    .join(
        existing_customer_keys_df,
        on="customer_id",
        how="left_anti"
    )
    .count()
)

missing_delete_key_count = (
    customer_deletes_df
    .select("customer_id")
    .join(
        existing_customer_keys_df,
        on="customer_id",
        how="left_anti"
    )
    .count()
)

conflicting_insert_key_count = (
    customer_inserts_df
    .select("customer_id")
    .join(
        existing_customer_keys_df,
        on="customer_id",
        how="inner"
    )
    .count()
)

operation_counts = {
    row["operation"]: row["count"]
    for row in (
        customer_changes_df
        .groupBy("operation")
        .count()
        .collect()
    )
}

max_initial_event_timestamp = (
    customers_initial_df
    .agg(
        F.max("event_timestamp")
        .alias("max_event_timestamp")
    )
    .first()["max_event_timestamp"]
)

min_change_event_timestamp = (
    customer_changes_df
    .agg(
        F.min("event_timestamp")
        .alias("min_event_timestamp")
    )
    .first()["min_event_timestamp"]
)

assert actual_event_count == EXPECTED_EVENT_COUNT
assert distinct_event_customer_count == EXPECTED_EVENT_COUNT

assert operation_counts.get("UPDATE") == UPDATE_COUNT
assert operation_counts.get("INSERT") == INSERT_COUNT
assert operation_counts.get("DELETE") == DELETE_COUNT

assert null_event_key_count == 0
assert duplicate_event_count == 0
assert missing_update_key_count == 0
assert missing_delete_key_count == 0
assert conflicting_insert_key_count == 0

assert (
    min_change_event_timestamp
    > max_initial_event_timestamp
)

print(f"Generated CDC events: {actual_event_count:,}")

print(
    "Distinct event customers: "
    f"{distinct_event_customer_count:,}"
)

print(f"Null event keys: {null_event_key_count:,}")
print(f"Duplicate events: {duplicate_event_count:,}")

print(
    "Missing UPDATE keys: "
    f"{missing_update_key_count:,}"
)

print(
    "Missing DELETE keys: "
    f"{missing_delete_key_count:,}"
)

print(
    "Conflicting INSERT keys: "
    f"{conflicting_insert_key_count:,}"
)

print(
    "Latest initial event: "
    f"{max_initial_event_timestamp}"
)

print(
    "Earliest change event: "
    f"{min_change_event_timestamp}"
)

display(
    customer_changes_df
    .groupBy("operation")
    .count()
    .orderBy("operation")
)

display(
    customer_changes_df
    .orderBy("event_timestamp", "customer_id")
    .limit(20)
)

## 7. Persist and Revalidate the Raw CDC Batch

Persist the validated CDC events as raw JSON files and read them back using the explicit customer schema to confirm that the stored batch is complete.

In [0]:
# The synthetic CDC batch is fully regenerated on every run.
(
    customer_changes_df.write
    .format("json")
    .mode("overwrite")
    .save(CUSTOMERS_CHANGE_BATCH_PATH)
)

saved_customer_changes_df = (
    spark.read
    .schema(CUSTOMER_SCHEMA)
    .json(CUSTOMERS_CHANGE_BATCH_PATH)
)

saved_event_count = (
    saved_customer_changes_df.count()
)

saved_operation_counts = {
    row["operation"]: row["count"]
    for row in (
        saved_customer_changes_df
        .groupBy("operation")
        .count()
        .collect()
    )
}

assert saved_event_count == EXPECTED_EVENT_COUNT
assert saved_operation_counts.get("UPDATE") == UPDATE_COUNT
assert saved_operation_counts.get("INSERT") == INSERT_COUNT
assert saved_operation_counts.get("DELETE") == DELETE_COUNT

print(f"Saved CDC events: {saved_event_count:,}")
print(f"Target path: {CUSTOMERS_CHANGE_BATCH_PATH}")

display(
    saved_customer_changes_df
    .groupBy("operation")
    .count()
    .orderBy("operation")
)